# Data Preprocessing

## Objective

The goal of this notebook is to transform the raw hospital readmission dataset into a clean, consistent, and machine-learning-ready dataset while preventing data leakage.

### Preprocessing Tasks

- Load the raw dataset and supporting metadata
- Convert missing value placeholders (`?`) to `NaN`
- Create the binary target variable
- Map administrative ID columns to meaningful categories
- Group ICD-9 diagnosis codes
- Group medical specialties
- Handle missing values
- Remove invalid records
- Perform a patient-level train/test split
- Save the processed datasets for downstream modeling

In [1]:
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", None)

RANDOM_STATE = 42

## 1. Load Raw Dataset

The project uses two files:

- **diabetic_data.csv** : Main hospital encounter dataset.
- **IDs_mapping.csv** : Lookup table that explains coded administrative variables such as admission type, discharge disposition and admission source.

In [2]:
import yaml

CONFIG_PATH =  '../configs/paths.yaml'

with open(CONFIG_PATH,"r") as file:
    paths = yaml.safe_load(file)

DATA_PATH = paths['raw_data']['diabetic_data']
MAPPING_PATH = paths['raw_data']['mapping_data']

df = pd.read_csv(DATA_PATH)
mapping = pd.read_csv(MAPPING_PATH)

print(f"Dataset Shape : {df.shape}")
print(f"Mapping Shape : {mapping.shape}")

Dataset Shape : (101766, 50)
Mapping Shape : (67, 2)


## 2. Initial Data Quality Assessment

Before applying any preprocessing, we inspect the dataset for common quality issues.

The following checks will be performed:

- Missing value encoding
- Duplicate rows
- Duplicate patients
- Data types

In [3]:
df.head()

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,?,Pediatrics-Endocrinology,41,0,1,0,0,0,250.83,?,?,1,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,?,?,59,0,18,0,0,0,276,250.01,255,9,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,?,?,11,5,13,2,0,1,648,250,V27,6,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,?,?,44,1,16,0,0,0,8,250.43,403,7,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,?,?,51,0,8,0,0,0,197,157,250,5,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [4]:
print("=" * 60)
print("Duplicate Rows")
print("=" * 60)

print(df.duplicated().sum())

print()

print("=" * 60)
print("Duplicate Patients")
print("=" * 60)

print(df["patient_nbr"].duplicated().sum())
display(
    df[
        [
            "diag_1",
            "diag_2",
            "diag_3",
        ]
    ].head(10)
)
print()

print("=" * 60)
print("Data Types")
print("=" * 60)

display(df.dtypes.value_counts())

Duplicate Rows
0

Duplicate Patients
30248


,diag_1,diag_2,diag_3
0,250.83,?,?
1,276,250.01,255
2,648,250,V27
3,8,250.43,403
4,197,157,250
5,414,411,250
6,414,411,V45
7,428,492,250
8,398,427,38
9,434,198,486



Data Types


str      37
int64    13
Name: count, dtype: int64

## Observation

- The dataset contains multiple encounters for the same patient.
- This confirms that a random row-wise train/test split would introduce data leakage.
- A patient-level split will therefore be performed later in the preprocessing pipeline.

## 3. Replace Missing Value Placeholders

The dataset represents missing values using the character **"?"** instead of proper `NaN` values.

Replacing these placeholders allows Pandas and Scikit-learn to correctly recognize missing data during preprocessing.

In [5]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from src.data.cleaning import (
    create_binary_target, 
    remove_columns, 
    missing_value_summary, 
    replace_missing_values
)

In [6]:
df = replace_missing_values(df, placeholder = '?')

## 4. Target Engineering

### Observation

The original target column **`readmitted`** contains three categories:

- **NO** : Patient was not readmitted.
- **>30** : Patient was readmitted after 30 days.
- **<30** : Patient was readmitted within 30 days.

### Decision

The objective of this project is to predict whether a patient is readmitted **within 30 days**.

Therefore, the target will be converted into a binary classification problem:

| Original | Binary |
|----------|--------|
| NO | 0 |
| >30 | 0 |
| <30 | 1 |

In [7]:
df = create_binary_target(df,source_column='readmitted', target_column= 'readmitted_binary')

display(df[["readmitted", "readmitted_binary"]].head())

,readmitted,readmitted_binary
0,NO,0
1,>30,0
2,NO,0
3,NO,0
4,NO,0


In [8]:
print("Original Target Distribution\n")

display(df["readmitted"].value_counts())

print("\nBinary Target Distribution\n")

display(df["readmitted_binary"].value_counts())

Original Target Distribution



readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64


Binary Target Distribution



readmitted_binary
0    90409
1    11357
Name: count, dtype: int64

### Verification

The target variable has been successfully converted into a binary classification target.

The original `readmitted` column is intentionally retained at this stage for traceability and easier debugging.

It will be removed later after all preprocessing steps have been completed.

In [9]:
display(missing_value_summary(df))

,Missing Count,Missing Percentage
weight,98569,96.86
max_glu_serum,96420,94.75
A1Cresult,84748,83.28
medical_specialty,49949,49.08
payer_code,40256,39.56
race,2273,2.23
diag_3,1423,1.40
diag_2,358,0.35
diag_1,21,0.02
patient_nbr,0,0.00


In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 51 columns):
 #   Column                    Non-Null Count   Dtype
---  ------                    --------------   -----
 0   encounter_id              101766 non-null  int64
 1   patient_nbr               101766 non-null  int64
 2   race                      99493 non-null   str  
 3   gender                    101766 non-null  str  
 4   age                       101766 non-null  str  
 5   weight                    3197 non-null    str  
 6   admission_type_id         101766 non-null  int64
 7   discharge_disposition_id  101766 non-null  int64
 8   admission_source_id       101766 non-null  int64
 9   time_in_hospital          101766 non-null  int64
 10  payer_code                61510 non-null   str  
 11  medical_specialty         51817 non-null   str  
 12  num_lab_procedures        101766 non-null  int64
 13  num_procedures            101766 non-null  int64
 14  num_medications           10176

## 5. Administrative Feature Analysis

### Observation

The dataset contains three administrative features stored as integer codes:

- `admission_type_id`
- `discharge_disposition_id`
- `admission_source_id`

These values are not ordinal. They are categorical identifiers whose meanings are
provided in the accompanying `IDs_mapping.csv` file.

### Decision

Instead of leaving these coded values unexplained, we will inspect the mapping file
to understand what each identifier represents before deciding whether to create
human-readable features.

In [11]:
mapping.head(20)

,admission_type_id,description
0,1,Emergency
1,2,Urgent
2,3,Elective
3,4,Newborn
4,5,Not Available
5,6,NaN
6,7,Trauma Center
7,8,Not Mapped
8,NaN,NaN
9,discharge_disposition_id,description


In [12]:
mapping.info()

<class 'pandas.DataFrame'>
RangeIndex: 67 entries, 0 to 66
Data columns (total 2 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   admission_type_id  65 non-null     str  
 1   description        62 non-null     str  
dtypes: str(2)
memory usage: 1.2 KB


### Observation

The mapping file contains lookup information for the administrative identifier
columns.

This metadata will be used during preprocessing to interpret hospital admission,
discharge, and referral information.

At this stage, no transformations are applied because the original identifier
columns remain suitable categorical variables for machine learning models.

In [13]:
from src.data.diagnosis import group_diagnosis_codes

In [14]:
diagnosis_columns = ["diag_1", "diag_2", "diag_3"]

for column in diagnosis_columns:
    print(f"\n{column}")
    print("-" * 40)
    print(f"Unique Codes : {df[column].nunique()}")


diag_1
----------------------------------------
Unique Codes : 716

diag_2
----------------------------------------
Unique Codes : 748

diag_3
----------------------------------------
Unique Codes : 789


### Observation

Each diagnosis column contains several hundred unique ICD-9 codes.

Using these raw codes directly would produce an extremely high-cardinality categorical
feature.

To improve model generalization and interpretability, the diagnosis codes will be
grouped into broader disease categories.

In [15]:
diagnosis_columns = ["diag_1", "diag_2", "diag_3"]

for column in diagnosis_columns:
    df[f"{column}_group"] = df[column].apply(group_diagnosis_codes)

In [16]:
display(
    df[
        [
            "diag_1",
            "diag_1_group",
            "diag_2",
            "diag_2_group",
            "diag_3",
            "diag_3_group",
        ]
    ].head(10)
)

,diag_1,diag_1_group,diag_2,diag_2_group,diag_3,diag_3_group
0,250.83,Diabetes,NaN,Unknown,NaN,Unknown
1,276,Other,250.01,Diabetes,255,Other
2,648,Other,250,Diabetes,V27,Other
3,8,Other,250.43,Diabetes,403,Circulatory
4,197,Neoplasms,157,Neoplasms,250,Diabetes
5,414,Circulatory,411,Circulatory,250,Diabetes
6,414,Circulatory,411,Circulatory,V45,Other
7,428,Circulatory,492,Respiratory,250,Diabetes
8,398,Circulatory,427,Circulatory,38,Other
9,434,Circulatory,198,Neoplasms,486,Respiratory


In [17]:
for column in ["diag_1_group", "diag_2_group", "diag_3_group"]:
    print(f"\n{column}")
    print("-" * 40)
    display(df[column].value_counts(normalize = True))


diag_1_group
----------------------------------------


diag_1_group
Circulatory        0.299088
Other              0.178567
Respiratory        0.141727
Digestive          0.093106
Diabetes           0.086050
Injury             0.068530
Genitourinary      0.050282
Musculoskeletal    0.048710
Neoplasms          0.033734
Unknown            0.000206
Name: proportion, dtype: float64


diag_2_group
----------------------------------------


diag_2_group
Circulatory        0.313278
Other              0.260922
Diabetes           0.125720
Respiratory        0.107059
Genitourinary      0.082306
Digestive          0.040976
Neoplasms          0.025028
Injury             0.023859
Musculoskeletal    0.017334
Unknown            0.003518
Name: proportion, dtype: float64


diag_3_group
----------------------------------------


diag_3_group
Circulatory        0.297801
Other              0.286884
Diabetes           0.168593
Respiratory        0.072303
Genitourinary      0.065641
Digestive          0.038618
Injury             0.019122
Musculoskeletal    0.018818
Neoplasms          0.018238
Unknown            0.013983
Name: proportion, dtype: float64

### Verification

The ICD-9 diagnosis codes have been successfully transformed into clinically meaningful
diagnostic categories.

The original diagnosis columns are retained to preserve the raw information, while the
new grouped features will be used during exploratory analysis and model development.

## 6. Medical Specialty Grouping

### Observation

The `medical_specialty` column contains dozens of unique specialties.

Many specialties appear only a handful of times, resulting in a high-cardinality
categorical feature that increases model complexity without providing significant
predictive value.

### Decision

Medical specialties will be grouped into broader clinical categories.

Rare specialties will be combined into an **Other** category to reduce feature
cardinality while preserving clinically meaningful information.

In [18]:
from src.data.specialty import group_medical_specialty

In [20]:
print(f"Unique Medical Specialties : {df['medical_specialty'].nunique()}")

display(
    df["medical_specialty"]
    .value_counts(dropna=False)
    .head(20)
)

Unique Medical Specialties : 72


medical_specialty
NaN                                  49949
InternalMedicine                     14635
Emergency/Trauma                      7565
Family/GeneralPractice                7440
Cardiology                            5352
Surgery-General                       3099
Nephrology                            1613
Orthopedics                           1400
Orthopedics-Reconstructive            1233
Radiologist                           1140
Pulmonology                            871
Psychiatry                             854
Urology                                685
ObstetricsandGynecology                671
Surgery-Cardiovascular/Thoracic        652
Gastroenterology                       564
Surgery-Vascular                       533
Surgery-Neuro                          468
PhysicalMedicineandRehabilitation      391
Oncology                               348
Name: count, dtype: int64

In [21]:
df["medical_specialty_group"] = (
    df["medical_specialty"]
    .apply(group_medical_specialty)
)

In [22]:
display(
    df[
        [
            "medical_specialty",
            "medical_specialty_group",
        ]
    ].sample(10, random_state=RANDOM_STATE)
)

,medical_specialty,medical_specialty_group
35956,InternalMedicine,Internal Medicine
60927,NaN,Missing
79920,NaN,Missing
50078,Gastroenterology,Internal Medicine
44080,NaN,Missing
4727,NaN,Missing
29944,Nephrology,Internal Medicine
84575,NaN,Missing
59479,InternalMedicine,Internal Medicine
56742,NaN,Missing


In [23]:
display(
    df["medical_specialty_group"]
    .value_counts(dropna=False)
)

medical_specialty_group
Missing                       49949
Internal Medicine             17803
Emergency/Critical Care        7568
General Practice               7440
Surgery                        7161
Cardiology                     5359
Other Specialty                3985
Mental/Neurological Health     1166
OB-GYN                          773
Pediatrics                      562
Name: count, dtype: int64

### Verification

Medical specialties have been successfully grouped into broader clinical
categories.

The grouped feature substantially reduces the number of unique categories while
retaining clinically relevant information.

The original `medical_specialty` column is preserved for traceability and will
be removed during feature selection prior to model training.

## 7. Missing Value Treatment

### Observation

The dataset contains several features with missing values. However, not every
missing feature should be handled in the same way.

Some variables contain almost no useful information due to extremely high
missingness, while others contain valuable clinical information and should
be retained.

### Decision Strategy

Each feature will be evaluated individually using three criteria:

1. Percentage of missing values.
2. Clinical importance.
3. Suitability for imputation.

Based on these criteria, features will either be:

- Dropped
- Imputed
- Retained

In [24]:
missing_summary = missing_value_summary(df)

display(missing_summary)

,Missing Count,Missing Percentage
weight,98569,96.86
max_glu_serum,96420,94.75
A1Cresult,84748,83.28
medical_specialty,49949,49.08
payer_code,40256,39.56
race,2273,2.23
diag_3,1423,1.40
diag_2,358,0.35
diag_1,21,0.02
gender,0,0.00


### Missing Value Decisions

| Feature | Missing (%) | Decision | Reason |
|----------|------------:|----------|--------|
| weight | ~96.9 | Drop | Too many missing values to impute reliably |
| max_glu_serum | ~94.8 | Drop | Extremely sparse clinical measurement |
| A1Cresult | ~83.3 | Drop | Very high missingness |
| medical_specialty | ~49.1 | Keep | Already grouped into broader categories |
| payer_code | ~39.6 | Keep | Missing values will be treated as "Missing" |
| race | ~2.2 | Keep | Missing values will be labeled as "Unknown" |
| diag_1 | ~0.02 | Keep | Very small amount of missing data |
| diag_2 | ~0.35 | Keep | Very small amount of missing data |
| diag_3 | ~1.40 | Keep | Very small amount of missing data |

### 7.1 Drop Features with Extremely High Missingness

The following features contain more than 80–90% missing values and are removed
from the dataset because reliable imputation is not feasible.

In [25]:
HIGH_MISSING_COLUMNS = [
    "weight",
    "max_glu_serum",
    "A1Cresult",
]

df = remove_columns(df, HIGH_MISSING_COLUMNS)

print("Dropped Columns")

for column in HIGH_MISSING_COLUMNS:
    print(f"✓ {column}")

Dropped Columns
✓ weight
✓ max_glu_serum
✓ A1Cresult


In [26]:
print(f"Current Dataset Shape : {df.shape}")

display(df.head())

Current Dataset Shape : (101766, 52)


,encounter_id,patient_nbr,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted,readmitted_binary,diag_1_group,diag_2_group,diag_3_group,medical_specialty_group
0,2278392,8222157,Caucasian,Female,[0-10),6,25,1,1,NaN,Pediatrics-Endocrinology,41,0,1,0,0,0,250.83,NaN,NaN,1,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO,0,Diabetes,Unknown,Unknown,Pediatrics
1,149190,55629189,Caucasian,Female,[10-20),1,1,7,3,NaN,NaN,59,0,18,0,0,0,276,250.01,255,9,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,>30,0,Other,Diabetes,Other,Missing
2,64410,86047875,AfricanAmerican,Female,[20-30),1,1,7,2,NaN,NaN,11,5,13,2,0,1,648,250,V27,6,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO,0,Other,Diabetes,Other,Missing
3,500364,82442376,Caucasian,Male,[30-40),1,1,7,2,NaN,NaN,44,1,16,0,0,0,8,250.43,403,7,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO,0,Other,Diabetes,Circulatory,Missing
4,16680,42519267,Caucasian,Male,[40-50),1,1,7,1,NaN,NaN,51,0,8,0,0,0,197,157,250,5,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO,0,Neoplasms,Neoplasms,Diabetes,Missing


### 7.2 Retain Remaining Missing Values

The remaining missing values are intentionally preserved.

These missing values will be handled later using Scikit-learn preprocessing
pipelines (`SimpleImputer`) during model training.

Keeping missing values in their original form ensures that:

- The processed dataset remains faithful to the original data.
- Imputation is learned only from the training data.
- The exact same preprocessing is applied during inference.
- Data leakage is avoided.

In [27]:
remaining_missing = (
    df.isna()
      .sum()
      .sort_values(ascending=False)
)

display(
    remaining_missing[remaining_missing > 0]
)

medical_specialty    49949
payer_code           40256
race                  2273
diag_3                1423
diag_2                 358
diag_1                  21
dtype: int64

### Verification

Only clinically useful features retain missing values.

These missing values will be imputed inside the Scikit-learn preprocessing
pipeline during model training.

No manual imputation has been performed at this stage.

## 8. Patient-Level Train/Test Split

### Observation

The dataset contains multiple encounters for the same patient, identified by
the `patient_nbr` column.

A conventional row-wise train/test split would allow encounters from the same
patient to appear in both the training and testing datasets, resulting in
data leakage.

### Decision

To ensure an unbiased evaluation, the dataset will be split at the **patient
level**, ensuring that each patient appears in only one subset.

In [28]:
from src.data.split import (
    patient_level_split,
    split_summary,
)

In [29]:
train_df, test_df = patient_level_split(
    df,
    patient_column="patient_nbr",
    test_size=0.20,
    random_state=RANDOM_STATE,
)

In [30]:
split_summary(
    train_df,
    test_df,
    patient_column="patient_nbr",
)

Patient-Level Train/Test Split Summary
Train Encounters : 81,477
Test Encounters  : 20,289

Train Patients   : 57,214
Test Patients    : 14,304

Data Leakage Detected : False


## 9. Feature Cleanup

### Observation

During preprocessing, several new features were created while the original
raw features were retained for traceability.

Before saving the processed dataset, columns that are no longer required for
modeling will be removed.

The patient identifier is retained only until the train/test split is complete
and is now removed to prevent data leakage during model training.

In [31]:
COLUMNS_TO_DROP = [
    "patient_nbr",
    "encounter_id",
    "readmitted",
    "diag_1",
    "diag_2",
    "diag_3",
    "medical_specialty",
]

train_df = train_df.drop(columns=COLUMNS_TO_DROP)
test_df = test_df.drop(columns=COLUMNS_TO_DROP)

print(f"Training Shape : {train_df.shape}")
print(f"Testing Shape  : {test_df.shape}")

Training Shape : (81477, 45)
Testing Shape  : (20289, 45)


In [32]:
print("Training Columns")

display(train_df.columns.to_frame(name="Feature"))

print("\nTesting Columns")

display(test_df.columns.to_frame(name="Feature"))

Training Columns


,Feature
race,race
gender,gender
age,age
admission_type_id,admission_type_id
discharge_disposition_id,discharge_disposition_id
admission_source_id,admission_source_id
time_in_hospital,time_in_hospital
payer_code,payer_code
num_lab_procedures,num_lab_procedures
num_procedures,num_procedures



Testing Columns


,Feature
race,race
gender,gender
age,age
admission_type_id,admission_type_id
discharge_disposition_id,discharge_disposition_id
admission_source_id,admission_source_id
time_in_hospital,time_in_hospital
payer_code,payer_code
num_lab_procedures,num_lab_procedures
num_procedures,num_procedures


### Verification

The processed datasets now contain:

- Binary target variable
- Grouped diagnosis features
- Grouped medical specialty feature
- Administrative identifier features
- Original numerical and categorical predictors

Identifier columns and raw diagnosis columns have been removed because they are
either non-predictive or replaced with engineered features.

## 10. Save Processed Dataset

The cleaned datasets are saved for downstream tasks, including exploratory data
analysis, baseline models, and Random Forest training.

Separating preprocessing from modeling ensures that every model is trained on
the exact same version of the data.

In [33]:
import os

os.makedirs("../data/processed", exist_ok=True)

train_df.to_csv(
    "../data/processed/train.csv",
    index=False,
)

test_df.to_csv(
    "../data/processed/test.csv",
    index=False,
)

print("✓ train.csv saved")
print("✓ test.csv saved")

✓ train.csv saved
✓ test.csv saved


In [34]:
print("Processed Training Dataset")

display(train_df.head())

print("\nProcessed Testing Dataset")

display(test_df.head())

Processed Training Dataset


,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted_binary,diag_1_group,diag_2_group,diag_3_group,medical_specialty_group
0,Caucasian,Female,[0-10),6,25,1,1,NaN,41,0,1,0,0,0,1,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,0,Diabetes,Unknown,Unknown,Pediatrics
1,Caucasian,Female,[10-20),1,1,7,3,NaN,59,0,18,0,0,0,9,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,0,Other,Diabetes,Other,Missing
2,AfricanAmerican,Female,[20-30),1,1,7,2,NaN,11,5,13,2,0,1,6,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,0,Other,Diabetes,Other,Missing
3,Caucasian,Male,[30-40),1,1,7,2,NaN,44,1,16,0,0,0,7,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,0,Other,Diabetes,Circulatory,Missing
4,Caucasian,Male,[50-60),2,1,2,3,NaN,31,6,16,0,0,0,9,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,Yes,0,Circulatory,Circulatory,Diabetes,Missing



Processed Testing Dataset


,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted_binary,diag_1_group,diag_2_group,diag_3_group,medical_specialty_group
0,Caucasian,Male,[40-50),1,1,7,1,NaN,51,0,8,0,0,0,5,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,0,Neoplasms,Neoplasms,Diabetes,Missing
1,Caucasian,Male,[60-70),3,1,2,4,NaN,70,1,21,0,0,0,7,Steady,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,0,Circulatory,Circulatory,Other,Missing
2,Caucasian,Female,[70-80),1,1,7,6,NaN,27,0,16,0,0,0,8,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,Yes,0,Injury,Injury,Diabetes,General Practice
3,Caucasian,Female,[60-70),1,1,7,2,NaN,41,0,11,0,0,0,6,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,Down,No,No,No,No,No,Ch,Yes,0,Circulatory,Diabetes,Circulatory,Cardiology
4,Caucasian,Male,[50-60),2,1,2,1,NaN,44,1,15,0,0,0,9,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,Yes,0,Injury,Circulatory,Diabetes,Missing


# Conclusion

The raw hospital dataset has been successfully transformed into a clean and
consistent dataset suitable for machine learning.

Completed preprocessing:

- Replaced missing value placeholders (`?`) with `NaN`
- Created a binary target variable
- Grouped ICD-9 diagnosis codes
- Grouped medical specialties
- Removed features with excessive missing values
- Preserved remaining missing values for pipeline-based imputation
- Performed a patient-level train/test split
- Removed identifier and redundant features
- Saved the processed datasets

---

## Next Notebook

[**03_exploratory_data_analysis.ipynb**](03_exploratory_data_analysis.ipynb)

The next notebook explores the processed training dataset to understand feature
distributions, target imbalance, and relationships between predictors and
hospital readmission.